In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_141426"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg
from src.validation import merges
from src.utils.logger import log_result

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "18_backtest_simulation"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Bibliotecas e parâmetros
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

paths = {
    "labels": os.path.join(PROC_PATH, "labels_y_daily.csv"),
    "oof": os.path.join(PROC_PATH, "16_oof_predictions.csv"),
}
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo obrigatório não encontrado: {path}")

with open(os.path.join(PROC_PATH, "results_16_models_tfidf.json"), "r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

STRATEGIES = [
    {"name": "long_only_55", "long_th": 0.55, "short_th": 0.45, "allow_short": False, "cost": 0.0005},
    {"name": "long_only_60", "long_th": 0.60, "short_th": 0.40, "allow_short": False, "cost": 0.0005},
    {"name": "long_short_sym", "long_th": 0.55, "short_th": 0.45, "allow_short": True, "cost": 0.0007},
]

print(json.dumps(STRATEGIES, indent=2))

[
  {
    "name": "long_only_55",
    "long_th": 0.55,
    "short_th": 0.45,
    "allow_short": false,
    "cost": 0.0005
  },
  {
    "name": "long_only_60",
    "long_th": 0.6,
    "short_th": 0.4,
    "allow_short": false,
    "cost": 0.0005
  },
  {
    "name": "long_short_sym",
    "long_th": 0.55,
    "short_th": 0.45,
    "allow_short": true,
    "cost": 0.0007
  }
]


In [4]:
# 3. Carregar dados e preparar base
labels = pd.read_csv(paths["labels"], parse_dates=["day"])
oof = pd.read_csv(paths["oof"], parse_dates=["day"])

oof = oof.drop(columns=["ret_next", "close"], errors="ignore")
labels_merge = labels.drop(columns=["y"], errors="ignore")

df = oof.merge(
    labels_merge,
    on="day",
    how="left",
)
if df.empty:
    raise ValueError("Sem previsoes para o backtest.")

if "ret_next" not in df.columns:
    rename_map = {}
    for cand in ["ret_next_y", "ret_next_label"]:
        if cand in df.columns:
            rename_map[cand] = "ret_next"
            break
    if rename_map:
        df.rename(columns=rename_map, inplace=True)
    else:
        raise ValueError("labels_y_daily.csv precisa ter a coluna ret_next.")

df.sort_values(["model", "day"], inplace=True)
df["sentiment"] = df["proba"] * 2 - 1

display(df.head())
print(df.groupby("model")["day"].nunique())


,model,fold,row_id_x,day,y,proba,row_id_y,ret_next,close,sentiment
0,logreg_l2,0,4,2025-10-02,0,0.509085,4,-0.005,NaN,0.018170
1,logreg_l2,0,5,2025-10-03,1,0.482862,5,0.005,NaN,-0.034277
2,logreg_l2,0,6,2025-10-06,0,0.496190,6,-0.005,NaN,-0.007619
3,logreg_l2,1,7,2025-10-07,1,0.430391,7,0.005,NaN,-0.139218
4,logreg_l2,1,8,2025-10-08,0,0.471600,8,-0.005,NaN,-0.056800


model
logreg_l2    12
rf_200       12
Name: day, dtype: int64


In [5]:
merges.check_intersection(oof, labels, col_left="day", col_right="day", min_days=90)

[day] 2025-10-02 → 2025-10-17 | 12 dias únicos | 24 registros | missing=0
[day] 2025-09-19 → 2025-10-17 | 16 dias únicos | 16 registros | missing=0
Dias em comum: 12


C:\Users\ander\AppData\Local\Temp\ipykernel_32996\1725920795.py:1: UserWarning: Amostra curta: apenas 12 dias em comum (< 90). Resultados estatísticos podem ficar instáveis.
  merges.check_intersection(oof, labels, col_left="day", col_right="day", min_days=90)


{'left': DateSummary(min_date=Timestamp('2025-10-02 00:00:00'), max_date=Timestamp('2025-10-17 00:00:00'), n_days=12, n_records=24, n_missing=0),
 'right': DateSummary(min_date=Timestamp('2025-09-19 00:00:00'), max_date=Timestamp('2025-10-17 00:00:00'), n_days=16, n_records=16, n_missing=0),
 'common_days': 12,
 'days_list': DatetimeIndex(['2025-10-02', '2025-10-03', '2025-10-06', '2025-10-07',
                '2025-10-08', '2025-10-09', '2025-10-10', '2025-10-13',
                '2025-10-14', '2025-10-15', '2025-10-16', '2025-10-17'],
               dtype='datetime64[ns]', freq='B')}

In [6]:
# 4. Funções utilitárias
from typing import Dict


def compute_metrics(returns: pd.Series, equity_curve: pd.Series) -> Dict[str, float]:
    if returns.empty:
        return {"cagr": np.nan, "vol": np.nan, "sharpe": np.nan, "max_dd": np.nan, "hit_ratio": np.nan}
    total_return = (1 + returns).prod() - 1
    years = len(returns) / 252
    cagr = (1 + total_return) ** (1 / years) - 1 if years > 0 else np.nan
    vol = returns.std(ddof=0) * np.sqrt(252)
    avg = returns.mean() * 252
    sharpe = avg / vol if vol and vol > 0 else np.nan
    hit_ratio = (returns > 0).mean()
    running_max = equity_curve.cummax()
    dd = equity_curve / running_max - 1
    max_dd = dd.min()
    return {
        "total_return": total_return,
        "cagr": cagr,
        "vol": vol,
        "sharpe": sharpe,
        "max_dd": max_dd,
        "hit_ratio": hit_ratio,
    }


def run_strategy(data: pd.DataFrame, config: Dict[str, float]) -> pd.DataFrame:
    df_strat = data.copy().reset_index(drop=True)
    long_th = config["long_th"]
    short_th = config["short_th"]
    allow_short = config.get("allow_short", True)
    cost = config.get("cost", 0.0005)

    raw_signal = np.where(
        df_strat["proba"] >= long_th,
        1,
        np.where(df_strat["proba"] <= short_th, -1 if allow_short else 0, 0),
    )

    df_strat["signal"] = raw_signal
    df_strat["turnover"] = np.abs(np.diff(np.insert(raw_signal, 0, 0)))
    df_strat["cost"] = df_strat["turnover"] * cost
    df_strat["strategy_ret"] = df_strat["signal"] * df_strat["ret_next"] - df_strat["cost"]
    df_strat["equity"] = (1 + df_strat["strategy_ret"]).cumprod()

    return df_strat

In [7]:
# 5. Rodar simulações para cada modelo x estratégia
perf_rows = []
equity_records = []
daily_records = []

for model_name, sub in df.groupby("model"):
    ordered = sub.sort_values("day").reset_index(drop=True)
    for config in STRATEGIES:
        strat_name = config["name"]
        result = run_strategy(ordered, config)
        metrics = compute_metrics(result["strategy_ret"], result["equity"])
        perf_rows.append({
            "model": model_name,
            "strategy": strat_name,
            "n_days": len(result),
            **metrics,
        })

        equity_records.append({
            "model": model_name,
            "strategy": strat_name,
            "day": result["day"],
            "equity": result["equity"],
        })
        tmp = result.copy()
        tmp["strategy"] = strat_name
        tmp["model"] = model_name
        daily_records.append(tmp)

perf_df = pd.DataFrame(perf_rows)
display(perf_df.sort_values(["model", "strategy"]))

,model,strategy,n_days,total_return,cagr,vol,sharpe,max_dd,hit_ratio
0,logreg_l2,long_only_55,12,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
1,logreg_l2,long_only_60,12,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
2,logreg_l2,long_short_sym,12,-0.012751,-0.236235,0.033143,-8.110378,-0.012751,0.000000
3,rf_200,long_only_55,12,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
4,rf_200,long_only_60,12,0.000000,0.000000,0.000000,NaN,0.000000,0.000000
5,rf_200,long_short_sym,12,0.002114,0.045340,0.060891,0.758735,-0.005750,0.333333


In [8]:
best = perf_df.sort_values("sharpe", ascending=False).groupby("model").head(1)
for _, row in best.iterrows():
    metrics = {"cagr": float(row.get("cagr", 0) or 0), "sharpe": float(row.get("sharpe", 0) or 0)}
    extra = {
        "dataset": "backtest_daily",
        "strategy": row.get("strategy"),
        "total_return": float(row.get("total_return", 0) or 0),
        "max_dd": float(row.get("max_dd", 0) or 0),
        "hit_ratio": float(row.get("hit_ratio", 0) or 0),
        "n_days": int(row.get("n_days", 0) or 0),
    }
    log_result(model_name=row.get("model"), dataset_name="backtest_daily", metrics=metrics, extra=extra)


C:\Users\ander\AppData\Roaming\Python\Python311\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


In [9]:
# 6. Gráfico de curvas de capital
if equity_records:
    traces = []
    for rec in equity_records:
        traces.append(
            go.Scatter(
                x=rec["day"],
                y=rec["equity"],
                mode="lines",
                name=f"{rec['model']} | {rec['strategy']}",
            )
        )
    equity_fig = go.Figure(data=traces)
    equity_fig.update_layout(title="Curvas de capital - Estratégias", yaxis_title="Equity (R$ 1 inicial)")
    equity_fig.show()
else:
    equity_fig = None

In [10]:
# 7. Salvar resultados
perf_file = os.path.join(PROC_PATH, "18_backtest_results.csv")
daily_file = os.path.join(PROC_PATH, "18_backtest_daily_curves.csv")
fig_file = os.path.join(PROC_PATH, "18_backtest_equity.html")

perf_df.to_csv(perf_file, index=False, encoding="utf-8")
if daily_records:
    daily_df = pd.concat(daily_records, ignore_index=True)
    daily_df.to_csv(daily_file, index=False, encoding="utf-8")
else:
    daily_df = pd.DataFrame()

if equity_fig is not None:
    equity_fig.write_html(fig_file)

print("Arquivos salvos:")
print(perf_file)
print(daily_file)
print(fig_file)

Arquivos salvos:
C:\Users\ander\OneDrive\TCC_USP\data_processed\18_backtest_results.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\18_backtest_daily_curves.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\18_backtest_equity.html


In [11]:
# 8. Resumo final
from IPython.display import Markdown

best = perf_df.sort_values("sharpe", ascending=False).head(3)
best_md = best.to_string(index=False)

md = "**18_backtest_simulation concluido**\n"
md += "- Estrategias testadas: " + ", ".join([c["name"] for c in STRATEGIES]) + "\n"
md += "- Best Sharpe (Top 3):\n" + best_md + "\n"
md += "\n**Proximo:** `20_final_dashboard_analysis` -> consolidar metricas e gerar painel final."

Markdown(md)


**18_backtest_simulation concluido**
- Estrategias testadas: long_only_55, long_only_60, long_short_sym
- Best Sharpe (Top 3):
    model       strategy  n_days  total_return      cagr      vol    sharpe    max_dd  hit_ratio
   rf_200 long_short_sym      12      0.002114  0.045340 0.060891  0.758735 -0.005750   0.333333
logreg_l2 long_short_sym      12     -0.012751 -0.236235 0.033143 -8.110378 -0.012751   0.000000
logreg_l2   long_only_55      12      0.000000  0.000000 0.000000       NaN  0.000000   0.000000

**Proximo:** `20_final_dashboard_analysis` -> consolidar metricas e gerar painel final.